# Naive Vector RAG Baseline
This notebook builds a minimal Naive Vector RAG: PDF extraction, chunking, embeddings (SentenceTransformer -> jinaai/jina-embeddings-v4), retrieval with PyTorch cosine similarity, and 2D projection for visualization. Exports JSON used by the React frontend.

In [14]:
# Environment / GPU check
import sys, os
import torch
print('Python', sys.version.split())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if device.type == 'cuda':
    try:
        print('CUDA device:', torch.cuda.get_device_name(0))
    except Exception:
        print('CUDA available')
else:
    print('Using CPU')
print('Embedding device ->', device)

Python ['3.10.20', '|', 'packaged', 'by', 'Anaconda,', 'Inc.', '|', '(main,', 'Mar', '11', '2026,', '17:42:35)', '[MSC', 'v.1942', '64', 'bit', '(AMD64)]']
CUDA device: NVIDIA GeForce RTX 4060 Laptop GPU
Embedding device -> cuda


## PDF Extraction

In [15]:
# Use the specific PDF in ./data and extract pages with PyMuPDF (fitz)
import os, fitz, pandas as pd
pdf_path = '../data/MachineLearningNotes.pdf'
assert os.path.exists(pdf_path), f'PDF not found: {pdf_path}'
doc = fitz.open(pdf_path)
pages = []
for i in range(len(doc)):
    text = doc[i].get_text().strip()
    pages.append({'page_number': i+1, 'text': text})
df_pages = pd.DataFrame(pages)
df_pages.head()

,page_number,text
0,1,Machine Learning\nAndrea Passerini\na.y. 2022/...
1,2,1\nThese notes are for the course ”Machine Lea...
2,3,Contents\n1\nIntroduction\n7\n1.1\nDesign of a...
3,4,CONTENTS\n3\n4.7\nPrincipal Component Analysis...
4,5,CONTENTS\n4\n8.4\nPractical suggestions for bu...


## Chunking

In [16]:
# Deterministic word-based chunking (~600 words, small overlap)
import re, uuid
def chunk_page_text(page_text, page_number, words_per_chunk=650, overlap=50):
    tokens = re.findall(r'\S+', page_text)
    chunks = []
    if not tokens:
        return chunks
    step = words_per_chunk - overlap
    for i in range(0, max(1, len(tokens)), step):
        chunk_tokens = tokens[i:i+words_per_chunk]
        if not chunk_tokens:
            break
        text = ' '.join(chunk_tokens).strip()
        chunk_id = str(uuid.uuid4())
        chunks.append({'chunk_id': chunk_id, 'page_number': page_number, 'chunk_text': text, 'word_count': len(chunk_tokens)})
        if i+words_per_chunk >= len(tokens):
            break
    return chunks
chunks = []
for r in pages:
    ch = chunk_page_text(r['text'], r['page_number'])
    chunks.extend(ch)
len(chunks)

262

## Embedding with SentenceTransformer (jina-embeddings-v4)

In [17]:
# Robust embedding model loading: prefer Jina v4, then v3, then a stable SBERT fallback
from sentence_transformers import SentenceTransformer
import torch

candidate_models = [
    ('jinaai/jina-embeddings-v4', True),
    ('jinaai/jina-embeddings-v3', True),
    ('sentence-transformers/all-MiniLM-L6-v2', False),
]

model = None
active_model_name = None
load_errors = []

for name, use_remote_code in candidate_models:
    try:
        model = SentenceTransformer(name, device=str(device), trust_remote_code=use_remote_code)
        active_model_name = name
        break
    except Exception as e:
        load_errors.append(f'{name}: {type(e).__name__}: {e}')

if model is None:
    print('Model loading attempts failed:')
    for err in load_errors:
        print('-', err)
    raise RuntimeError('No embedding model could be loaded. Check internet/cache and package versions.')

print('Loaded embedding model ->', active_model_name)

def embed_texts(texts, batch_size=64):
    embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        emb = model.encode(batch, convert_to_tensor=True, normalize_embeddings=True)
        embeddings.append(emb)
    if len(embeddings) == 0:
        return torch.empty((0, model.get_sentence_embedding_dimension()), device=device)
    return torch.cat(embeddings, dim=0)

# Prepare texts and compute embeddings (kept as torch tensors)
all_texts = [c['chunk_text'] for c in chunks]
emb_t = embed_texts(all_texts)
# Ensure embeddings are on the requested device
emb_t = emb_t.to(device)
print('Embeddings computed; device ->', emb_t.device, 'shape ->', emb_t.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4888.96it/s]


Loaded embedding model -> sentence-transformers/all-MiniLM-L6-v2
Embeddings computed; device -> cuda:0 shape -> torch.Size([262, 384])


In [18]:
# Verification cell: print embeddings device and shape
print('embeddings.device =', emb_t.device)
print('embeddings.shape =', emb_t.shape)

embeddings.device = cuda:0
embeddings.shape = torch.Size([262, 384])


## Cosine Similarity Retrieval

In [19]:
# Sample query, embed, compute cosine similarities with PyTorch
query = 'What is the main contribution of the paper?'
q_t = model.encode([query], convert_to_tensor=True, normalize_embeddings=True)
q_t = q_t.to(device)
# Compute cosine similarities using matmul (both tensors normalized)
sims = torch.matmul(emb_t, q_t[0])
sims = sims.cpu().numpy()
topk = 5
idxs = sims.argsort()[::-1][:topk]
results = []
for rank, idx in enumerate(idxs, 1):
    c = chunks[idx]
    results.append({'rank': rank, 'chunk_id': c['chunk_id'], 'page_number': c['page_number'], 'similarity_score': float(sims[idx]), 'chunk_text_preview': c['chunk_text'][:300], 'full_chunk_text': c['chunk_text']})
results

[{'rank': 1,
  'chunk_id': '65eb3363-c694-4203-8137-44203a2f7e6a',
  'page_number': 2,
  'similarity_score': 0.3382169008255005,
  'chunk_text_preview': '1 These notes are for the course ”Machine Learning” held by professor An- drea Passerini. These notes do not directly cover the laboratory part; still, while discussing the theory many concepts applicable to the laboratory part might be covered. We tried to include most of the materials used during ',
  'full_chunk_text': '1 These notes are for the course ”Machine Learning” held by professor An- drea Passerini. These notes do not directly cover the laboratory part; still, while discussing the theory many concepts applicable to the laboratory part might be covered. We tried to include most of the materials used during the lectures or given as reference. Our aim was to create notes that allow to get a clear overview of all the topics and to arrange them in a structured manner, while also being fairly exhaustive. We tried our best to avo

## 2D Embedding Projection

In [20]:
# Try UMAP, fallback to PCA
import numpy as np
try:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=42)
    pts = reducer.fit_transform(emb_t.cpu().numpy())
except Exception:
    from sklearn.decomposition import PCA
    reducer = PCA(n_components=2)
    pts = reducer.fit_transform(emb_t.cpu().numpy())

# Project query into same space if reducer supports transform
try:
    q2 = reducer.transform(q_t.cpu().numpy())
except Exception:
    q2 = q_t.cpu().numpy()

# Build visualization dict
vis_points = []
for i, c in enumerate(chunks):
    vis_points.append({'chunk_id': c['chunk_id'], 'page_number': c['page_number'], 'x': float(pts[i,0]), 'y': float(pts[i,1]), 'preview': c['chunk_text'][:200], 'is_retrieved': c['chunk_id'] in [r['chunk_id'] for r in results], 'is_cited': False, 'similarity_score': float(sims[i])})
query_point = {'x': float(q2[0,0]), 'y': float(q2[0,1]), 'is_query': True, 'query': query}

## Export for React Visualization

In [21]:
import json, os

# Write exports for backend/analysis use
os.makedirs('../backend_or_exports', exist_ok=True)
# Write exports used directly by the React app
os.makedirs('../frontend/public/data', exist_ok=True)

chunks_out = [
    {
        'chunk_id': c['chunk_id'],
        'page_number': c['page_number'],
        'chunk_text': c['chunk_text'],
        'word_count': c['word_count'],
        'preview': c['chunk_text'][:200]
    }
    for c in chunks
]

query_out = {
    'query': query,
    'results': results,
    'query_embedding': q_t[0].cpu().numpy().tolist()
}

# Visualization JSON (includes 2D coords and flags)
vis = {'points': vis_points, 'query_point': query_point}

# Backend exports
with open('../backend_or_exports/naive_rag_chunks.json', 'w', encoding='utf-8') as f:
    json.dump({'chunks': chunks_out}, f, ensure_ascii=False, indent=2)
with open('../backend_or_exports/naive_rag_query_result.json', 'w', encoding='utf-8') as f:
    json.dump(query_out, f, ensure_ascii=False, indent=2)
with open('../backend_or_exports/naive_rag_vis.json', 'w', encoding='utf-8') as f:
    json.dump(vis, f, ensure_ascii=False, indent=2)

# Frontend data mirror
with open('../frontend/public/data/naive_rag_chunks.json', 'w', encoding='utf-8') as f:
    json.dump({'chunks': chunks_out}, f, ensure_ascii=False, indent=2)
with open('../frontend/public/data/naive_rag_query_result.json', 'w', encoding='utf-8') as f:
    json.dump(query_out, f, ensure_ascii=False, indent=2)
with open('../frontend/public/data/naive_rag_vis.json', 'w', encoding='utf-8') as f:
    json.dump(vis, f, ensure_ascii=False, indent=2)

print('Wrote exports to ../backend_or_exports/ and ../frontend/public/data/')

Wrote exports to ../backend_or_exports/ and ../frontend/public/data/
